# Robustness and error analysis

This notebook profiles diagnostic errors across ablation conditions and summarizes section-removal perturbation results. Exported case-level tables contain identifiers and diagnosis labels, but exclude clinical text, prompts, and model reasoning.

In [ ]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "clinical_cds").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / "output/.matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_DIR = REPO_ROOT / "output/notebook_artifacts/robustness_error"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE_ORDER = [
    "direct",
    "flat_rag",
    "graph_rag",
    "structured_argument",
    "symbolic_argument",
]
STATUS_ORDER = ["correct", "incorrect_covered", "abstained", "execution_error"]
STATUS_COLORS = {
    "correct": "#2A7F62",
    "incorrect_covered": "#D17A22",
    "abstained": "#687078",
    "execution_error": "#A33A2B",
}

def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else REPO_ROOT / path

def parse_path_list(value):
    specs = []
    for item in value.split(os.pathsep):
        if not item.strip():
            continue
        if "=" in item:
            label, path_value = item.split("=", 1)
        else:
            path_value = item
            label = Path(path_value).name
        specs.append((label.strip(), resolve_path(path_value.strip())))
    return specs

multiple = os.environ.get("EXPERIMENT_DIRS")
single = os.environ.get("EXPERIMENT_DIR")
if multiple:
    experiment_specs = parse_path_list(multiple)
elif single:
    experiment_specs = [(Path(single).name, resolve_path(single))]
else:
    experiment_specs = [
        ("direct_test", REPO_ROOT / "output/experiments/direct_test"),
        ("direct_test_strict", REPO_ROOT / "output/experiments/direct_test_strict"),
        ("direct_test_umls", REPO_ROOT / "output/experiments/direct_test_umls"),
        ("medqa_test", REPO_ROOT / "output/experiments/medqa_test"),
    ]

experiment_specs = [
    (label, path)
    for label, path in experiment_specs
    if (path / "evaluation/case_metrics.csv").is_file()
]
if not experiment_specs:
    raise FileNotFoundError(
        "No evaluated experiment was found. Run the experiments in the README or set "
        "EXPERIMENT_DIR to a completed run."
    )

experiment_specs

## Outcome status by condition

In [ ]:
case_frames = []
for run_label, run_dir in experiment_specs:
    rows = pd.read_csv(run_dir / "evaluation/case_metrics.csv")
    rows["run"] = run_label
    case_frames.append(rows)
case_metrics = pd.concat(case_frames, ignore_index=True)

def outcome_status(row):
    if float(row["error"]) > 0:
        return "execution_error"
    if float(row["covered"]) == 0:
        return "abstained"
    if float(row["exact_match"]) == 1:
        return "correct"
    return "incorrect_covered"

case_metrics["outcome_status"] = case_metrics.apply(outcome_status, axis=1)
error_profile = (
    case_metrics.groupby(["run", "dataset", "mode", "outcome_status"], as_index=False)
    .agg(case_count=("case_id", "count"))
)
error_profile["case_fraction"] = error_profile["case_count"] / error_profile.groupby(
    ["run", "dataset", "mode"]
)["case_count"].transform("sum")
error_profile.to_csv(OUTPUT_DIR / "outcome_status_by_mode.csv", index=False)
error_profile

In [ ]:
profile_plot = error_profile.pivot_table(
    index=["run", "dataset", "mode"],
    columns="outcome_status",
    values="case_fraction",
    fill_value=0.0,
).reset_index()
profile_plot["mode_order"] = profile_plot["mode"].map(
    {mode: index for index, mode in enumerate(MODE_ORDER)}
)
profile_plot = profile_plot.sort_values(["run", "dataset", "mode_order"])

figure, axis = plt.subplots(figsize=(10.0, max(4.0, 0.42 * len(profile_plot))))
left = np.zeros(len(profile_plot))
for status in STATUS_ORDER:
    values = profile_plot[status].to_numpy() if status in profile_plot else np.zeros(len(profile_plot))
    axis.barh(
        np.arange(len(profile_plot)),
        values,
        left=left,
        color=STATUS_COLORS[status],
        label=status.replace("_", " "),
    )
    left += values
axis.set_yticks(np.arange(len(profile_plot)))
axis.set_yticklabels(
    [f"{row.run}, {row.dataset}: {row.mode}" for row in profile_plot.itertuples()]
)
axis.set_xlim(0, 1)
axis.set_xlabel("Case fraction")
axis.set_title("Outcome composition by diagnostic condition")
axis.grid(axis="x", color="#D8D8D8", linewidth=0.6)
axis.set_axisbelow(True)
axis.legend(frameon=False, ncol=2)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "outcome_status_by_mode.png", dpi=220)
plt.show()

## Paired error transitions

Transitions show whether each added component corrects or introduces errors on the same cases.

In [ ]:
TRANSITIONS = [
    ("retrieval_context", "direct", "flat_rag"),
    ("graph_topology", "flat_rag", "graph_rag"),
    ("structured_argumentation", "graph_rag", "structured_argument"),
    ("symbolic_resolution", "structured_argument", "symbolic_argument"),
    ("full_method", "direct", "symbolic_argument"),
]
transition_rows = []

for (run, dataset_name), rows in case_metrics.groupby(["run", "dataset"], sort=False):
    paired = rows.pivot_table(
        index="case_id",
        columns="mode",
        values="exact_match",
        aggfunc="first",
    )
    for effect, baseline_mode, comparison_mode in TRANSITIONS:
        if baseline_mode not in paired or comparison_mode not in paired:
            continue
        values = paired[[baseline_mode, comparison_mode]].dropna()
        if values.empty:
            continue
        categories = np.select(
            [
                (values[baseline_mode] == 0) & (values[comparison_mode] == 1),
                (values[baseline_mode] == 1) & (values[comparison_mode] == 0),
                (values[baseline_mode] == 1) & (values[comparison_mode] == 1),
            ],
            ["improved", "degraded", "stable_correct"],
            default="stable_incorrect",
        )
        counts = pd.Series(categories).value_counts()
        for transition in ("improved", "degraded", "stable_correct", "stable_incorrect"):
            count = int(counts.get(transition, 0))
            transition_rows.append(
                {
                    "run": run,
                    "dataset": dataset_name,
                    "effect": effect,
                    "baseline_mode": baseline_mode,
                    "comparison_mode": comparison_mode,
                    "transition": transition,
                    "case_count": count,
                    "case_fraction": count / len(values),
                    "paired_n": len(values),
                }
            )

transition_columns = [
    "run",
    "dataset",
    "effect",
    "baseline_mode",
    "comparison_mode",
    "transition",
    "case_count",
    "case_fraction",
    "paired_n",
]
mode_transitions = pd.DataFrame(transition_rows, columns=transition_columns)
mode_transitions.to_csv(OUTPUT_DIR / "paired_error_transitions.csv", index=False)
mode_transitions

In [ ]:
if mode_transitions.empty:
    transition_balance = pd.DataFrame(
        columns=["run", "dataset", "effect", "improved", "degraded"]
    )
else:
    transition_balance = (
        mode_transitions[mode_transitions["transition"].isin(["improved", "degraded"])]
        .pivot_table(
            index=["run", "dataset", "effect"],
            columns="transition",
            values="case_fraction",
            fill_value=0.0,
        )
        .reset_index()
    )
for column in ("improved", "degraded"):
    if column not in transition_balance:
        transition_balance[column] = 0.0

figure, axis = plt.subplots(figsize=(9.0, max(3.5, 0.5 * max(len(transition_balance), 1))))
if transition_balance.empty:
    axis.text(0.5, 0.5, "No paired mode transitions were available.", ha="center", va="center")
    axis.set_axis_off()
else:
    positions = np.arange(len(transition_balance))
    axis.barh(positions - 0.18, transition_balance["improved"], height=0.34, color="#2A7F62", label="improved")
    axis.barh(positions + 0.18, transition_balance["degraded"], height=0.34, color="#A33A2B", label="degraded")
    axis.set_yticks(positions)
    axis.set_yticklabels(
        [f"{row.run}, {row.dataset}: {row.effect.replace('_', ' ')}" for row in transition_balance.itertuples()]
    )
    axis.set_xlabel("Case fraction")
    axis.grid(axis="x", color="#D8D8D8", linewidth=0.6)
    axis.set_axisbelow(True)
    axis.legend(frameon=False)
axis.set_title("Paired corrections and regressions")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "paired_error_transitions.png", dpi=220)
plt.show()

## Text-free diagnostic error catalogue

In [ ]:
prediction_rows = []
for run_label, run_dir in experiment_specs:
    predictions_path = run_dir / "predictions.jsonl"
    if not predictions_path.is_file():
        continue
    with predictions_path.open("r", encoding="utf-8") as source:
        for line in source:
            if not line.strip():
                continue
            payload = json.loads(line)
            metadata = payload.get("metadata") or {}
            retrieved_diagnoses = sorted(
                {
                    str(fact.get("diagnosis_label") or "")
                    for fact in metadata.get("retrieved_facts", [])
                    if fact.get("diagnosis_label")
                }
            )
            prediction_rows.append(
                {
                    "run": run_label,
                    "case_id": str(payload.get("case_id") or ""),
                    "dataset": str(payload.get("dataset") or ""),
                    "mode": str(payload.get("mode") or ""),
                    "gold_label": str(payload.get("gold_label") or ""),
                    "predicted_label": str(payload.get("predicted_label") or ""),
                    "abstained": bool(payload.get("abstained")),
                    "error_present": payload.get("error") is not None,
                    "argument_count": metadata.get("argument_count", 0),
                    "argument_schemes": "; ".join(metadata.get("argument_schemes", [])),
                    "argument_schema_validity": metadata.get("argument_schema_validity"),
                    "argument_evidence_validity": metadata.get("argument_evidence_validity"),
                    "verifier_review_coverage": metadata.get("verifier_review_coverage"),
                    "reasoner_preferred_diagnosis": metadata.get("reasoner_preferred_diagnosis", ""),
                    "symbolic_selected_diagnosis": metadata.get("symbolic_selected_diagnosis", ""),
                    "argument_resolution_changed": metadata.get("argument_resolution_changed"),
                    "resolver_id": metadata.get("resolver_id", ""),
                    "retrieved_diagnoses": "; ".join(retrieved_diagnoses),
                }
            )

prediction_index = pd.DataFrame(prediction_rows)
metric_columns = ["run", "case_id", "dataset", "mode", "exact_match", "covered", "citation_validity", "retrieval_gold_coverage"]
if prediction_index.empty:
    error_catalogue = pd.DataFrame(
        columns=[
            "run",
            "case_id",
            "dataset",
            "mode",
            "gold_label",
            "predicted_label",
            "abstained",
            "error_present",
            "argument_count",
            "argument_schemes",
            "argument_schema_validity",
            "argument_evidence_validity",
            "verifier_review_coverage",
            "reasoner_preferred_diagnosis",
            "symbolic_selected_diagnosis",
            "argument_resolution_changed",
            "resolver_id",
            "retrieved_diagnoses",
            "exact_match",
            "covered",
            "citation_validity",
            "retrieval_gold_coverage",
        ]
    )
else:
    error_catalogue = prediction_index.merge(
        case_metrics[metric_columns],
        on=["run", "case_id", "dataset", "mode"],
        how="left",
    )
    error_catalogue = error_catalogue[
        (error_catalogue["exact_match"] != 1)
        | error_catalogue["abstained"]
        | error_catalogue["error_present"]
    ]

error_catalogue.to_csv(OUTPUT_DIR / "diagnostic_error_catalogue.csv", index=False)
error_catalogue.head(20)

## Argument trace visualization

This view follows one case through the reasoner proposal, verifier decisions and attacks, and symbolic resolution. Set `TRACE_CASE_ID` before starting Jupyter to select a case; otherwise the first complete trace is used.

In [ ]:
from IPython.display import Image, display

from clinical_cds.trace_visualization import load_argument_trace, plot_argument_trace

trace_case_id = os.environ.get("TRACE_CASE_ID")
trace_plot_path = None
for run_label, run_dir in experiment_specs:
    traces_path = run_dir / "argument_traces.jsonl"
    if not traces_path.is_file():
        continue
    try:
        selected_trace = load_argument_trace(traces_path, case_id=trace_case_id)
    except KeyError:
        continue
    selected_case_id = str(selected_trace.get("case_id") or "unknown")
    trace_plot_path = plot_argument_trace(
        selected_trace,
        OUTPUT_DIR / f"argument_trace_{run_label}_{selected_case_id}.png",
    )
    display(Image(filename=str(trace_plot_path)))
    break

if trace_plot_path is None:
    print("No complete argument trace was found for the requested case.")

## Section-removal robustness

The perturbation run removes the clinical section containing the largest number of annotated observations. If the perturbation artifact is absent, this section exports empty schemas and identifies the CLI command required to generate it.

In [ ]:
perturbation_multiple = os.environ.get("PERTURBATION_DIRS")
perturbation_single = os.environ.get("PERTURBATION_DIR")
if perturbation_multiple:
    perturbation_specs = parse_path_list(perturbation_multiple)
elif perturbation_single:
    perturbation_specs = [(Path(perturbation_single).name, resolve_path(perturbation_single))]
else:
    perturbation_specs = [
        ("direct_section_removal", REPO_ROOT / "output/experiments/direct_section_removal")
    ]

perturbation_frames = []
for run_label, run_dir in perturbation_specs:
    metrics_path = run_dir / "evaluation/perturbation_metrics.csv"
    if not metrics_path.is_file() or metrics_path.stat().st_size == 0:
        continue
    rows = pd.read_csv(metrics_path)
    rows["run"] = run_label
    perturbation_frames.append(rows)

perturbation_columns = [
    "run",
    "mode",
    "pair_count",
    "base_accuracy",
    "perturbed_accuracy",
    "accuracy_difference",
    "answer_change_rate",
    "stale_citation_rate",
    "base_abstention_rate",
    "perturbed_abstention_rate",
]
section_columns = [
    "run",
    "mode",
    "removed_section",
    "pair_count",
    "base_accuracy",
    "perturbed_accuracy",
    "answer_change_rate",
    "stale_citation_rate",
]

if perturbation_frames:
    perturbation_metrics = pd.concat(perturbation_frames, ignore_index=True)
    perturbation_summary = (
        perturbation_metrics.groupby(["run", "mode"], as_index=False)
        .agg(
            pair_count=("base_case_id", "count"),
            base_accuracy=("base_correct", "mean"),
            perturbed_accuracy=("perturbed_correct", "mean"),
            answer_change_rate=("answer_changed", "mean"),
            stale_citation_rate=("stale_removed_section_citation", "mean"),
            base_abstention_rate=("base_abstained", "mean"),
            perturbed_abstention_rate=("perturbed_abstained", "mean"),
        )
    )
    perturbation_summary["accuracy_difference"] = (
        perturbation_summary["perturbed_accuracy"] - perturbation_summary["base_accuracy"]
    )
    perturbation_summary = perturbation_summary[perturbation_columns]
    section_summary = (
        perturbation_metrics.groupby(["run", "mode", "removed_section"], as_index=False)
        .agg(
            pair_count=("base_case_id", "count"),
            base_accuracy=("base_correct", "mean"),
            perturbed_accuracy=("perturbed_correct", "mean"),
            answer_change_rate=("answer_changed", "mean"),
            stale_citation_rate=("stale_removed_section_citation", "mean"),
        )
    )[section_columns]
else:
    perturbation_metrics = pd.DataFrame()
    perturbation_summary = pd.DataFrame(columns=perturbation_columns)
    section_summary = pd.DataFrame(columns=section_columns)
    print(
        "No perturbation artifact was found. Run `python -m clinical_cds run-perturbations` "
        "as documented in the README, then rerun this notebook."
    )

perturbation_summary.to_csv(OUTPUT_DIR / "section_removal_summary.csv", index=False)
section_summary.to_csv(OUTPUT_DIR / "section_removal_by_section.csv", index=False)
perturbation_summary

In [ ]:
if not perturbation_summary.empty:
    labels = [f"{row.run}: {row.mode}" for row in perturbation_summary.itertuples()]
    positions = np.arange(len(perturbation_summary))
    figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
    axes[0].bar(
        positions - 0.18,
        perturbation_summary["base_accuracy"],
        width=0.36,
        color="#3478A5",
        label="base",
    )
    axes[0].bar(
        positions + 0.18,
        perturbation_summary["perturbed_accuracy"],
        width=0.36,
        color="#D17A22",
        label="section removed",
    )
    axes[0].set_ylabel("Exact diagnosis accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Accuracy under section removal")
    axes[0].legend(frameon=False)

    rate_width = 0.24
    for index, (column, label, color) in enumerate(
        [
            ("answer_change_rate", "answer changed", "#3478A5"),
            ("stale_citation_rate", "stale citation", "#A33A2B"),
            ("perturbed_abstention_rate", "abstained", "#687078"),
        ]
    ):
        axes[1].bar(
            positions + (index - 1) * rate_width,
            perturbation_summary[column],
            width=rate_width,
            color=color,
            label=label,
        )
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Response to removed evidence")
    axes[1].legend(frameon=False)

    for axis in axes:
        axis.set_xticks(positions)
        axis.set_xticklabels(labels, rotation=20, ha="right")
        axis.grid(axis="y", color="#D8D8D8", linewidth=0.6)
        axis.set_axisbelow(True)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "section_removal_robustness.png", dpi=220)
    plt.show()

The transition and perturbation analyses are descriptive paired analyses. Error cases should be interpreted against retrieval coverage, abstention, and citation validity rather than diagnosis accuracy alone.